In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
from huggingface_hub import snapshot_download
from transformers import TimesFmModel, TimesFmConfig
import torch
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from collections import Counter
from collections import defaultdict
import hdbscan
from momentfm import MOMENTPipeline

/opt/miniconda3/envs/timeSZ/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = ".././Datasets/Labeled_Data_Without_GPS/USA/"
file_paths = glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)

files_df = pd.DataFrame({
    "full_path": file_paths,
    "filename": [os.path.basename(p) for p in file_paths],
})

files_df["surface_id"] = files_df["filename"].str.extract(r"SurfaceTypeID_(\d+)").astype("Int64")

files_df.head(2) 

,full_path,filename,surface_id
0,.././Datasets/Labeled_Data_Without_GPS/USA/Sur...,2019-02-27_SurfaceTypeID_9_SamsungGalaxyJ7_exp...,9
1,.././Datasets/Labeled_Data_Without_GPS/USA/Sur...,2019-09-02_SurfaceTypeID_9_SamsungGalaxyS7_exp...,9


In [3]:
from tqdm import tqdm

# Windowing rows into fixed-size segments for model input

window_size = 416
step_percentage = 0.5
step = int(window_size * step_percentage)


windows = []
labels = []

for _, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Processing files"):
    path = row["full_path"]
    sid = row["surface_id"]
    df_current = pd.read_csv(path)

    # Filter for accelerometer data
    acc_df = df_current[df_current["sensorName"] == "Accelerometer"]

    if acc_df.shape[0] < window_size:
        continue
    
    try:
        acc_vals = acc_df[["valueX", "valueY", "valueZ"]].to_numpy()
        magnitude = np.linalg.norm(acc_vals, axis=1)
    except KeyError:
        print(f"Missing columns in {path}")
        continue
    
    for start in range(0, len(magnitude) - window_size + 1, step):
        window = magnitude[start:start + window_size]

        windows.append(window)
        labels.append(sid)

print(f"Total windows: {len(windows)}")

Processing files:  49%|████▉     | 319/645 [00:05<00:11, 28.06it/s] 

Missing columns in .././Datasets/Labeled_Data_Without_GPS/USA/SurfaceTypeID_8/2019-09-03_SurfaceTypeID_8_SamsungGalaxyJ7_exp6_subject2.csv


Processing files: 100%|██████████| 645/645 [00:12<00:00, 51.85it/s] 

Total windows: 67823


In [4]:
print(windows[0].shape)

(416,)


# Model part

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Load MOMENT — only one size currently available
pipeline = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={"task_name": "embedding"}   # set task to embedding extraction
)
pipeline.init()
model = pipeline.model if hasattr(pipeline, "model") else pipeline
model = model.to(device)
model.eval()

Using device: mps


Loading weights: 0it [00:00, ?it/s]
TimesFmModel LOAD REPORT from: google/timesfm-2.0-500m-pytorch
Key                                             | Status     | 
------------------------------------------------+------------+-
decoder.layers.{0...49}.self_attn.scaling       | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.v_proj.bias   | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.k_proj.bias   | UNEXPECTED | 
decoder.layers.{0...49}.mlp.gate_proj.bias      | UNEXPECTED | 
decoder.layers.{0...49}.mlp.down_proj.bias      | UNEXPECTED | 
decoder.layers.{0...49}.mlp.down_proj.weight    | UNEXPECTED | 
decoder.layers.{0...49}.input_layernorm.weight  | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.q_proj.bias   | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.o_proj.weight | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.q_proj.weight | UNEXPECTED | 
decoder.layers.{0...49}.self_attn.k_proj.weight | UNEXPECTED | 
decoder.layers.{0...49}.mlp.gate_proj.weight    | UNEXPECTED | 
decod

TimesFmModel(
  (input_ff_layer): TimesFmResidualBlock(
    (input_layer): Linear(in_features=64, out_features=1280, bias=True)
    (activation): SiLU()
    (output_layer): Linear(in_features=1280, out_features=1280, bias=True)
    (residual_layer): Linear(in_features=64, out_features=1280, bias=True)
  )
  (freq_emb): Embedding(3, 1280)
  (layers): ModuleList(
    (0-49): 50 x TimesFmDecoderLayer(
      (self_attn): TimesFmAttention(
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (o_proj): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (mlp): TimesFmMLP(
        (gate_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (down_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (layer_norm): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
      )

In [ ]:
def extract_embeddings_moment(pipeline, windows, batch_size=32):

    all_embeddings = []

    for i in tqdm(range(0, len(windows), batch_size), desc="Extracting embeddings"):

        batch = windows[i:i + batch_size]  # list of 1D arrays (B, T)

        # MOMENT expects (B, n_channels, T) — univariate so n_channels=1
        batch_tensor = torch.tensor(
            np.array(batch), dtype=torch.float32
        ).unsqueeze(1).to(device)  # (B, 1, 416)

        # input_mask: 1 = real data, 0 = padding — all real here
        input_mask = torch.ones(
            batch_tensor.shape[0], batch_tensor.shape[2],
            dtype=torch.long
        ).to(device)  # (B, T)

        with torch.no_grad():
            output = pipeline(
                x_enc=batch_tensor,
                input_mask=input_mask
            )

        # output.embeddings shape: (B, hidden_dim)  — already pooled!
        embeddings = output.embeddings

        all_embeddings.append(embeddings.detach().cpu().numpy())

    return np.vstack(all_embeddings)


In [ ]:
embeddings = extract_embeddings_moment(pipeline, windows, batch_size=32)
print("Embedding shape:", embeddings.shape)

  0%|          | 0/2120 [00:00<?, ?it/s]/var/folders/_t/trdgwqmj5vl_d8t1_8mx10d40000gn/T/ipykernel_11566/3092662860.py:9: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  batch_tensor = torch.tensor(batch, dtype=torch.float32).to(device)
 21%|██▏       | 453/2120 [03:23<10:33,  2.63it/s]

# Clustering

In [ ]:
labels = [l - 1 for l in labels]

In [ ]:
from collections import Counter

label_counts = Counter(labels)

print("Label Distribution:")
for label in sorted(label_counts.keys()):
    count = label_counts[label]
    print(f"Label {label + 1}: {count} samples")

print(f"\nTotal samples: {sum(label_counts.values())}")

Label Distribution:
Label 1: 970 samples
Label 2: 8373 samples
Label 3: 5676 samples
Label 4: 5334 samples
Label 5: 9437 samples
Label 6: 10593 samples
Label 7: 11085 samples
Label 8: 10216 samples
Label 9: 2996 samples
Label 10: 3143 samples

Total samples: 67823


In [ ]:
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

## K-means Clustering

In [ ]:
num_clusters = len(np.unique(labels))

kmeans = KMeans(n_clusters=num_clusters, random_state=42)
clusters = kmeans.fit_predict(embeddings_scaled)

In [ ]:
cluster_surface_map = {}

labels_array = np.array(labels)

for cluster_id in range(num_clusters):
    
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")


Cluster → Surface Mapping:
Cluster 0 → Surface 5
Cluster 1 → Surface 8
Cluster 2 → Surface 5
Cluster 3 → Surface 7
Cluster 4 → Surface 7
Cluster 5 → Surface 8
Cluster 6 → Surface 7
Cluster 7 → Surface 8
Cluster 8 → Surface 6
Cluster 9 → Surface 6


## HB Scan Clustering

In [ ]:
# Create HDBSCAN model
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,      # minimum points to form a cluster
    min_samples=None,         # can tune for density sensitivity
    metric='euclidean'
)

# Fit and predict
clusters = clusterer.fit_predict(embeddings_scaled)

In [ ]:
cluster_surface_map = {}

labels_array = np.array(labels)

# Get unique clusters (excluding noise = -1)
unique_clusters = [c for c in np.unique(clusters) if c != -1]

for cluster_id in unique_clusters:
    
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    
    # Skip empty clusters just in case
    if len(cluster_labels) == 0:
        continue
        
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")

# Optional: count noise points
noise_count = np.sum(clusters == -1)
print(f"\nNoise points: {noise_count}")


Cluster → Surface Mapping:
Cluster 0 → Surface 10
Cluster 1 → Surface 10
Cluster 2 → Surface 9
Cluster 3 → Surface 5

Noise points: 31795


## DB-Scan

In [ ]:
from sklearn.cluster import DBSCAN

clusterer = DBSCAN(
    eps=0.5,              # neighborhood radius — tune this carefully
    min_samples=5,
    metric='euclidean'
)

clusters = clusterer.fit_predict(embeddings_scaled)

cluster_surface_map = {}
labels_array = np.array(labels)
unique_clusters = [c for c in np.unique(clusters) if c != -1]

for cluster_id in unique_clusters:
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")

noise_count = np.sum(clusters == -1)
print(f"\nNoise points: {noise_count}")

## Balanced K-Means

In [ ]:
# pip install k-means-constrained
from k_means_constrained import KMeansConstrained

n_clusters = len(np.unique(labels))  # or set manually
size_min = len(embeddings_scaled) // (n_clusters * 2)
size_max = len(embeddings_scaled) // n_clusters * 2

clusterer = KMeansConstrained(
    n_clusters=n_clusters,
    size_min=size_min,
    size_max=size_max,
    random_state=42
)

clusters = clusterer.fit_predict(embeddings_scaled)

cluster_surface_map = {}
labels_array = np.array(labels)

for cluster_id in np.unique(clusters):
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")

## Spectral Clustering


In [ ]:
from sklearn.cluster import SpectralClustering

n_clusters = len(np.unique(labels))  # or set manually

clusterer = SpectralClustering(
    n_clusters=n_clusters,
    affinity='rbf',          # 'nearest_neighbors' is faster for large data
    gamma=1.0,
    random_state=42,
    n_jobs=-1
)

clusters = clusterer.fit_predict(embeddings_scaled)

# No noise points in spectral clustering
cluster_surface_map = {}
labels_array = np.array(labels)

for cluster_id in np.unique(clusters):
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")

## GNN-based Clustering

In [ ]:
# pip install torch torch-geometric
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import KMeans

# --- Step 1: Build k-NN graph from embeddings ---
k = 10
A = kneighbors_graph(embeddings_scaled, k, mode='connectivity', include_self=False)
edge_index = torch.tensor(np.array(A.nonzero()), dtype=torch.long)
x = torch.tensor(embeddings_scaled, dtype=torch.float)

data = Data(x=x, edge_index=edge_index)

# --- Step 2: Define GCN encoder ---
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# --- Step 3: Train with self-supervised loss (feature reconstruction) ---
in_dim = embeddings_scaled.shape[1]
model = GCNEncoder(in_dim, 64, 32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(100):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    # Reconstruction loss: encourage similar neighbors to have similar embeddings
    loss = F.mse_loss(out[edge_index[0]], out[edge_index[1]])
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# --- Step 4: Cluster GNN embeddings with KMeans ---
model.eval()
with torch.no_grad():
    gnn_embeddings = model(data.x, data.edge_index).numpy()

n_clusters = len(np.unique(labels))
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(gnn_embeddings)

cluster_surface_map = {}
labels_array = np.array(labels)

for cluster_id in np.unique(clusters):
    indices = np.where(clusters == cluster_id)[0]
    cluster_labels = labels_array[indices]
    if len(cluster_labels) == 0:
        continue
    dominant_surface = Counter(cluster_labels).most_common(1)[0][0]
    cluster_surface_map[cluster_id] = dominant_surface

print("\nCluster → Surface Mapping:")
for k, v in cluster_surface_map.items():
    print(f"Cluster {k} → Surface {v + 1}")